In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!git clone https://github.com/RIA-lab/MMTD.git

In [ ]:
%cd /kaggle/working/MMTD/DATA


In [ ]:
!pip install -U gdown

In [ ]:
!gdown 1w4HxNf099lQ41mhMyXzbGwI429qBGvbY

In [ ]:
!unzip /kaggle/working/MMTD/DATA/email_data.zip -d /kaggle/working/MMTD/DATA/

In [ ]:
import sys
import os
import torch
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from transformers import Trainer, TrainingArguments, BertTokenizerFast, BeitImageProcessor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# 1. Kendi mimari kodlarını DEĞİL, orijinal makale kodlarını tanıtıyoruz
sys.path.append('/kaggle/working/MMTD')
# Dikkat: MMTD_Advanced yerine orijinal MMTD'yi import ediyoruz!
from models import MMTD 
from Email_dataset import EDPDataset

# =====================================================================
# 🚨 1. GÜRÜLTÜ VE VERİ YÜKLEYİCİ (Birebir Aynı Şartlar)
# =====================================================================
def add_gaussian_noise(image, mean=0, std=25):
    img_array = np.array(image)
    noise = np.random.normal(mean, std, img_array.shape).astype(np.uint8)
    noisy_image_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_image_array)

def add_text_noise(text, noise_level=0.1):
    if noise_level <= 0 or noise_level > 1: return text
    noisy_text = list(text)
    num_noise_chars = int(len(text) * noise_level)
    for _ in range(num_noise_chars):
        index = random.randint(0, len(text) - 1)
        noisy_text[index] = random.choice("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890!@#$%^&*()_-+=[]{}|;:,.<>?")
    return ''.join(noisy_text)

class EDPCollatorRobust:
    def __init__(self, text_noise_level=0.0, image_noise=False):
        self.text_noise_level = text_noise_level
        self.image_noise = image_noise
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.feature_extractor = BeitImageProcessor.from_pretrained('microsoft/dit-base')

    def __call__(self, data):
        text_list, picture_list, label_list = zip(*data)
        
        if self.text_noise_level > 0:
            text_list = [add_text_noise(t, self.text_noise_level) for t in text_list]
        if self.image_noise:
            picture_list = [add_gaussian_noise(p) for p in picture_list]

        text_tensor = self.tokenizer(list(text_list), return_tensors='pt', max_length=256, truncation=True, padding='max_length')
        pixel_values = self.feature_extractor(list(picture_list), return_tensors='pt')
        labels = torch.LongTensor(label_list)
        
        inputs = dict()
        inputs.update(text_tensor)
        inputs.update(pixel_values)
        inputs['labels'] = labels
        return inputs

# =====================================================================
# 📊 2. GELİŞMİŞ METRİK HESAPLAYICI 
# =====================================================================
def compute_advanced_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)
    
    return {
        "accuracy": acc,
        "f1_score": f1,
        "precision": precision,
        "recall": recall
    }

# =====================================================================
# 🚀 3. TEST LABORATUVARI (RAKİP MODEL: FOLD 4)
# =====================================================================
if __name__ == "__main__":
    print("🚨 RAKİP MODEL (Orijinal MMTD - Fold 4) Detaylı Testi Başlıyor...\n")

    DATA_CSV_YOLU = '/kaggle/working/MMTD/DATA/email_data/EDP.csv' 
    RESIMLERIN_YOLU = '/kaggle/working/MMTD/DATA/email_data/pics'

    df = pd.read_csv(DATA_CSV_YOLU)
    df = df.rename(columns={"texts": "text", "pics": "image", "labels": "label"})
    _, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
    test_dataset = EDPDataset(RESIMLERIN_YOLU, test_df)

    # 2. Orijinal MMTD Modelini Kuruyoruz
    model = MMTD(bert_pretrain_weight='bert-base-multilingual-cased', 
                 beit_pretrain_weight='microsoft/dit-base')
    
    # 3. Sağdaki Kaggle Input Klasöründen Fold 4 Ağırlıklarını Çekiyoruz
    # Not: Kaggle klasör isimlerini küçük harfe çevirir. Eğer dosya adı 'pytorch_model.bin' değilse burayı ona göre güncelle.
    rakip_model_path = '/kaggle/input/mmtdcheckpointsfold4/pytorch_model.bin'
    
    print(f"🧠 Ağırlıklar yükleniyor: {rakip_model_path}")
    
    # Safetensors yerine standart PyTorch (torch.load) kullanıyoruz
    model.load_state_dict(torch.load(rakip_model_path))
    model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    model.eval()

    args = TrainingArguments(
        output_dir='./test_results_rakip',
        per_device_eval_batch_size=8,
        report_to="none"
    )

    scenarios = [
        {"name": "Temiz Veri", "text_noise": 0.0, "image_noise": False},
        {"name": "Görüntü Bozuk", "text_noise": 0.0, "image_noise": True},
        {"name": "Metin Bozuk", "text_noise": 0.1, "image_noise": False},
        {"name": "Tam Kaos", "text_noise": 0.1, "image_noise": True},
    ]

    names, accuracies, f1_scores = [], [], []

    print("\n📊 --- ORİJİNAL MMTD KARŞILAŞTIRMALI SONUÇLAR ---")
    for s in scenarios:
        collator = EDPCollatorRobust(text_noise_level=s["text_noise"], image_noise=s["image_noise"])
        trainer = Trainer(
            model=model,
            args=args,
            eval_dataset=test_dataset,
            data_collator=collator,
            compute_metrics=compute_advanced_metrics 
        )
        
        result = trainer.evaluate() 
        acc = result['eval_accuracy'] * 100
        f1 = result['eval_f1_score'] * 100
        
        names.append(s['name'])
        accuracies.append(acc)
        f1_scores.append(f1)
        
        print(f"❌ [{s['name']}] -> Acc: %{acc:.2f} | F1: %{f1:.2f} | Prec: %{result['eval_precision']*100:.2f} | Rec: %{result['eval_recall']*100:.2f}")

    # =====================================================================
    # 🎨 4. RAKİP MODEL İÇİN GRAFİK ÇİZİMİ (Kırmızı Tonlar)
    # =====================================================================
    print("\n🎨 Makale için grafik çiziliyor...")
    x = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))
    # Orijinal modeli seninkinden ayırt etmek için kırmızı ve turuncu renkler kullanıldı
    rects1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#d62728') 
    rects2 = ax.bar(x + width/2, f1_scores, width, label='F1-Score', color='#ff7f0e') 

    ax.set_ylabel('Yüzde (%)', fontsize=12, fontweight='bold')
    ax.set_title('Orijinal MMTD Modeli Gürbüzlük (Robustness) Performansı', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=11)
    ax.set_ylim([0, 110])
    ax.legend(loc='lower left')

    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.1f}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), 
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=10)

    autolabel(rects1)
    autolabel(rects2)
    plt.tight_layout()
    
    plt.savefig('/kaggle/working/mmtd_original_robustness.png', dpi=300)
    plt.show()
    print("✅ Grafik '/kaggle/working/mmtd_original_robustness.png' olarak kaydedildi! İndirebilirsin.")

In [ ]:
# Orijinal repodaki eski kütüphane isimlerini güncelleyen yama kodu
dosya_yolu = '/kaggle/working/MMTD/Email_dataset.py'

with open(dosya_yolu, 'r') as file:
    icerik = file.read()

# Eski isimleri yenileriyle değiştiriyoruz
icerik = icerik.replace('BeitFeatureExtractor', 'BeitImageProcessor')
icerik = icerik.replace('ViltFeatureExtractor', 'ViltImageProcessor')
icerik = icerik.replace('CLIPFeatureExtractor', 'CLIPImageProcessor')

with open(dosya_yolu, 'w') as file:
    file.write(icerik)

print("✅ Email_dataset.py başarıyla güncellendi! Artık test kodunu çalıştırabilirsin.")

In [ ]:
import sys
import os
import torch
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from transformers import Trainer, TrainingArguments, BertTokenizerFast, BeitImageProcessor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# 1. Kendi mimari kodlarını DEĞİL, orijinal makale kodlarını tanıtıyoruz
sys.path.append('/kaggle/working/MMTD')
# Dikkat: MMTD_Advanced yerine orijinal MMTD'yi import ediyoruz!
from models import MMTD 
from Email_dataset import EDPDataset

# =====================================================================
# 🚨 1. GÜRÜLTÜ VE VERİ YÜKLEYİCİ (Birebir Aynı Şartlar)
# =====================================================================
def add_gaussian_noise(image, mean=0, std=25):
    img_array = np.array(image)
    noise = np.random.normal(mean, std, img_array.shape).astype(np.uint8)
    noisy_image_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_image_array)

def add_text_noise(text, noise_level=0.1):
    if noise_level <= 0 or noise_level > 1: return text
    noisy_text = list(text)
    num_noise_chars = int(len(text) * noise_level)
    for _ in range(num_noise_chars):
        index = random.randint(0, len(text) - 1)
        noisy_text[index] = random.choice("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890!@#$%^&*()_-+=[]{}|;:,.<>?")
    return ''.join(noisy_text)

class EDPCollatorRobust:
    def __init__(self, text_noise_level=0.0, image_noise=False):
        self.text_noise_level = text_noise_level
        self.image_noise = image_noise
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.feature_extractor = BeitImageProcessor.from_pretrained('microsoft/dit-base')

    def __call__(self, data):
        text_list, picture_list, label_list = zip(*data)
        
        if self.text_noise_level > 0:
            text_list = [add_text_noise(t, self.text_noise_level) for t in text_list]
        if self.image_noise:
            picture_list = [add_gaussian_noise(p) for p in picture_list]

        text_tensor = self.tokenizer(list(text_list), return_tensors='pt', max_length=256, truncation=True, padding='max_length')
        pixel_values = self.feature_extractor(list(picture_list), return_tensors='pt')
        labels = torch.LongTensor(label_list)
        
        inputs = dict()
        inputs.update(text_tensor)
        inputs.update(pixel_values)
        inputs['labels'] = labels
        return inputs

# =====================================================================
# 📊 2. GELİŞMİŞ METRİK HESAPLAYICI 
# =====================================================================
def compute_advanced_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)
    
    return {
        "accuracy": acc,
        "f1_score": f1,
        "precision": precision,
        "recall": recall
    }

# =====================================================================
# 🚀 3. TEST LABORATUVARI (RAKİP MODEL: FOLD 4)
# =====================================================================
if __name__ == "__main__":
    print("🚨 RAKİP MODEL (Orijinal MMTD - Fold 4) Detaylı Testi Başlıyor...\n")

    DATA_CSV_YOLU = '/kaggle/working/MMTD/DATA/email_data/EDP.csv' 
    RESIMLERIN_YOLU = '/kaggle/working/MMTD/DATA/email_data/pics'

    df = pd.read_csv(DATA_CSV_YOLU)
    df = df.rename(columns={"texts": "text", "pics": "image", "labels": "label"})
    _, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
    test_dataset = EDPDataset(RESIMLERIN_YOLU, test_df)

    # 2. Orijinal MMTD Modelini Kuruyoruz
    model = MMTD(bert_pretrain_weight='bert-base-multilingual-cased', 
                 beit_pretrain_weight='microsoft/dit-base')
    
    # 3. Sağdaki Kaggle Input Klasöründen Fold 4 Ağırlıklarını Çekiyoruz
    # Not: Kaggle klasör isimlerini küçük harfe çevirir. Eğer dosya adı 'pytorch_model.bin' değilse burayı ona göre güncelle.
    rakip_model_path = '/kaggle/input/mmtdcheckpointsfold4/pytorch_model.bin'
    
    print(f"🧠 Ağırlıklar yükleniyor: {rakip_model_path}")
    
    # Safetensors yerine standart PyTorch (torch.load) kullanıyoruz
    model.load_state_dict(torch.load(rakip_model_path))
    model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    model.eval()

    args = TrainingArguments(
        output_dir='./test_results_rakip',
        per_device_eval_batch_size=8,
        report_to="none"
    )

    scenarios = [
        {"name": "Temiz Veri", "text_noise": 0.0, "image_noise": False},
        {"name": "Görüntü Bozuk", "text_noise": 0.0, "image_noise": True},
        {"name": "Metin Bozuk", "text_noise": 0.1, "image_noise": False},
        {"name": "Tam Kaos", "text_noise": 0.1, "image_noise": True},
    ]

    names, accuracies, f1_scores = [], [], []

    print("\n📊 --- ORİJİNAL MMTD KARŞILAŞTIRMALI SONUÇLAR ---")
    for s in scenarios:
        collator = EDPCollatorRobust(text_noise_level=s["text_noise"], image_noise=s["image_noise"])
        trainer = Trainer(
            model=model,
            args=args,
            eval_dataset=test_dataset,
            data_collator=collator,
            compute_metrics=compute_advanced_metrics 
        )
        
        result = trainer.evaluate() 
        acc = result['eval_accuracy'] * 100
        f1 = result['eval_f1_score'] * 100
        
        names.append(s['name'])
        accuracies.append(acc)
        f1_scores.append(f1)
        
        print(f"❌ [{s['name']}] -> Acc: %{acc:.2f} | F1: %{f1:.2f} | Prec: %{result['eval_precision']*100:.2f} | Rec: %{result['eval_recall']*100:.2f}")

    # =====================================================================
    # 🎨 4. RAKİP MODEL İÇİN GRAFİK ÇİZİMİ (Kırmızı Tonlar)
    # =====================================================================
    print("\n🎨 Makale için grafik çiziliyor...")
    x = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))
    # Orijinal modeli seninkinden ayırt etmek için kırmızı ve turuncu renkler kullanıldı
    rects1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#d62728') 
    rects2 = ax.bar(x + width/2, f1_scores, width, label='F1-Score', color='#ff7f0e') 

    ax.set_ylabel('Yüzde (%)', fontsize=12, fontweight='bold')
    ax.set_title('Orijinal MMTD Modeli Gürbüzlük (Robustness) Performansı', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=11)
    ax.set_ylim([0, 110])
    ax.legend(loc='lower left')

    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.1f}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), 
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=10)

    autolabel(rects1)
    autolabel(rects2)
    plt.tight_layout()
    
    plt.savefig('/kaggle/working/mmtd_original_robustness.png', dpi=300)
    plt.show()
    print("✅ Grafik '/kaggle/working/mmtd_original_robustness.png' olarak kaydedildi! İndirebilirsin.")

In [ ]:
# Orijinal repodaki eski kütüphane isimlerini temizleyen KAPSAMLI yama kodu
dosyalar = [
    '/kaggle/working/MMTD/Email_dataset.py',
    '/kaggle/working/MMTD/utils.py'
]

for dosya_yolu in dosyalar:
    try:
        with open(dosya_yolu, 'r') as file:
            icerik = file.read()

        # Eski isimleri yenileriyle değiştiriyoruz
        icerik = icerik.replace('BeitFeatureExtractor', 'BeitImageProcessor')
        icerik = icerik.replace('ViltFeatureExtractor', 'ViltImageProcessor')
        icerik = icerik.replace('CLIPFeatureExtractor', 'CLIPImageProcessor')

        with open(dosya_yolu, 'w') as file:
            file.write(icerik)
        print(f"✅ {dosya_yolu.split('/')[-1]} başarıyla güncellendi!")
    except FileNotFoundError:
        print(f"⚠️ Dosya bulunamadı: {dosya_yolu}")

print("🚀 Tüm yamalar tamamlandı! Şimdi o büyük test kodunu tekrar çalıştırabilirsin.")

In [ ]:
!pip install torchtext

In [ ]:
# Orijinal repodaki bozuk ve kullanılmayan 'torchtext' importlarını silen Cerrahi Yama
dosyalar = [
    '/kaggle/working/MMTD/Email_dataset.py',
    '/kaggle/working/MMTD/utils.py'
]

for dosya_yolu in dosyalar:
    try:
        with open(dosya_yolu, 'r') as file:
            satirlar = file.readlines()

        # İçinde 'torchtext' kelimesi geçmeyen satırları tut (geçenleri sil)
        yeni_satirlar = [satir for satir in satirlar if 'torchtext' not in satir]

        with open(dosya_yolu, 'w') as file:
            file.writelines(yeni_satirlar)
            
        print(f"✅ {dosya_yolu.split('/')[-1]} dosyasındaki bozuk importlar temizlendi!")
    except FileNotFoundError:
        print(f"⚠️ Dosya bulunamadı: {dosya_yolu}")

print("🚀 Operasyon tamam! Şimdi asıl test kodunu tekrar çalıştırabilirsin.")

In [ ]:
import sys
import os
import torch
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from transformers import Trainer, TrainingArguments, BertTokenizerFast, BeitImageProcessor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# 1. Kendi mimari kodlarını DEĞİL, orijinal makale kodlarını tanıtıyoruz
sys.path.append('/kaggle/working/MMTD')
# Dikkat: MMTD_Advanced yerine orijinal MMTD'yi import ediyoruz!
from models import MMTD 
from Email_dataset import EDPDataset

# =====================================================================
# 🚨 1. GÜRÜLTÜ VE VERİ YÜKLEYİCİ (Birebir Aynı Şartlar)
# =====================================================================
def add_gaussian_noise(image, mean=0, std=25):
    img_array = np.array(image)
    noise = np.random.normal(mean, std, img_array.shape).astype(np.uint8)
    noisy_image_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_image_array)

def add_text_noise(text, noise_level=0.1):
    if noise_level <= 0 or noise_level > 1: return text
    noisy_text = list(text)
    num_noise_chars = int(len(text) * noise_level)
    for _ in range(num_noise_chars):
        index = random.randint(0, len(text) - 1)
        noisy_text[index] = random.choice("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890!@#$%^&*()_-+=[]{}|;:,.<>?")
    return ''.join(noisy_text)

class EDPCollatorRobust:
    def __init__(self, text_noise_level=0.0, image_noise=False):
        self.text_noise_level = text_noise_level
        self.image_noise = image_noise
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.feature_extractor = BeitImageProcessor.from_pretrained('microsoft/dit-base')

    def __call__(self, data):
        text_list, picture_list, label_list = zip(*data)
        
        if self.text_noise_level > 0:
            text_list = [add_text_noise(t, self.text_noise_level) for t in text_list]
        if self.image_noise:
            picture_list = [add_gaussian_noise(p) for p in picture_list]

        text_tensor = self.tokenizer(list(text_list), return_tensors='pt', max_length=256, truncation=True, padding='max_length')
        pixel_values = self.feature_extractor(list(picture_list), return_tensors='pt')
        labels = torch.LongTensor(label_list)
        
        inputs = dict()
        inputs.update(text_tensor)
        inputs.update(pixel_values)
        inputs['labels'] = labels
        return inputs

# =====================================================================
# 📊 2. GELİŞMİŞ METRİK HESAPLAYICI 
# =====================================================================
def compute_advanced_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)
    
    return {
        "accuracy": acc,
        "f1_score": f1,
        "precision": precision,
        "recall": recall
    }

# =====================================================================
# 🚀 3. TEST LABORATUVARI (RAKİP MODEL: FOLD 4)
# =====================================================================
if __name__ == "__main__":
    print("🚨 RAKİP MODEL (Orijinal MMTD - Fold 4) Detaylı Testi Başlıyor...\n")

    DATA_CSV_YOLU = '/kaggle/working/MMTD/DATA/email_data/EDP.csv' 
    RESIMLERIN_YOLU = '/kaggle/working/MMTD/DATA/email_data/pics'

    df = pd.read_csv(DATA_CSV_YOLU)
    df = df.rename(columns={"texts": "text", "pics": "image", "labels": "label"})
    _, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
    test_dataset = EDPDataset(RESIMLERIN_YOLU, test_df)

    # 2. Orijinal MMTD Modelini Kuruyoruz
    model = MMTD(bert_pretrain_weight='bert-base-multilingual-cased', 
                 beit_pretrain_weight='microsoft/dit-base')
    
    # 3. Sağdaki Kaggle Input Klasöründen Fold 4 Ağırlıklarını Çekiyoruz
    # Not: Kaggle klasör isimlerini küçük harfe çevirir. Eğer dosya adı 'pytorch_model.bin' değilse burayı ona göre güncelle.
    rakip_model_path = '/kaggle/input/datasets/arafkubraa/mmtdcheckpointsfold4/pytorch_model.bin'
    print(f"🧠 Ağırlıklar yükleniyor: {rakip_model_path}")
    
    # Safetensors yerine standart PyTorch (torch.load) kullanıyoruz
    model.load_state_dict(torch.load(rakip_model_path))
    model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    model.eval()

    args = TrainingArguments(
        output_dir='./test_results_rakip',
        per_device_eval_batch_size=8,
        report_to="none"
    )

    scenarios = [
        {"name": "Temiz Veri", "text_noise": 0.0, "image_noise": False},
        {"name": "Görüntü Bozuk", "text_noise": 0.0, "image_noise": True},
        {"name": "Metin Bozuk", "text_noise": 0.1, "image_noise": False},
        {"name": "Tam Kaos", "text_noise": 0.1, "image_noise": True},
    ]

    names, accuracies, f1_scores = [], [], []

    print("\n📊 --- ORİJİNAL MMTD KARŞILAŞTIRMALI SONUÇLAR ---")
    for s in scenarios:
        collator = EDPCollatorRobust(text_noise_level=s["text_noise"], image_noise=s["image_noise"])
        trainer = Trainer(
            model=model,
            args=args,
            eval_dataset=test_dataset,
            data_collator=collator,
            compute_metrics=compute_advanced_metrics 
        )
        
        result = trainer.evaluate() 
        acc = result['eval_accuracy'] * 100
        f1 = result['eval_f1_score'] * 100
        
        names.append(s['name'])
        accuracies.append(acc)
        f1_scores.append(f1)
        
        print(f"❌ [{s['name']}] -> Acc: %{acc:.2f} | F1: %{f1:.2f} | Prec: %{result['eval_precision']*100:.2f} | Rec: %{result['eval_recall']*100:.2f}")

    # =====================================================================
    # 🎨 4. RAKİP MODEL İÇİN GRAFİK ÇİZİMİ (Kırmızı Tonlar)
    # =====================================================================
    print("\n🎨 Makale için grafik çiziliyor...")
    x = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))
    # Orijinal modeli seninkinden ayırt etmek için kırmızı ve turuncu renkler kullanıldı
    rects1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#d62728') 
    rects2 = ax.bar(x + width/2, f1_scores, width, label='F1-Score', color='#ff7f0e') 

    ax.set_ylabel('Yüzde (%)', fontsize=12, fontweight='bold')
    ax.set_title('Orijinal MMTD Modeli Gürbüzlük (Robustness) Performansı', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=11)
    ax.set_ylim([0, 110])
    ax.legend(loc='lower left')

    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.1f}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), 
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=10)

    autolabel(rects1)
    autolabel(rects2)
    plt.tight_layout()
    
    plt.savefig('/kaggle/working/mmtd_original_robustness.png', dpi=300)
    plt.show()
    print("✅ Grafik '/kaggle/working/mmtd_original_robustness.png' olarak kaydedildi! İndirebilirsin.")

In [ ]:
import sys
import os
import torch
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import Trainer, TrainingArguments, BertTokenizerFast, BeitImageProcessor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# =====================================================================
# 🛠 1. MİMARİ VE VERİ HAZIRLIĞI
# =====================================================================
sys.path.append('/kaggle/working/MMTD')
# Dikkat: MMTD_Advanced yerine orijinal MMTD'yi import ediyoruz
from models import MMTD 
from Email_dataset import EDPDataset

def add_gaussian_noise(image, mean=0, std=25):
    img_array = np.array(image)
    noise = np.random.normal(mean, std, img_array.shape).astype(np.uint8)
    noisy_image_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_image_array)

def add_text_noise(text, noise_level=0.1):
    if noise_level <= 0 or noise_level > 1: return text
    noisy_text = list(text)
    num_noise_chars = int(len(text) * noise_level)
    for _ in range(num_noise_chars):
        index = random.randint(0, len(text) - 1)
        noisy_text[index] = random.choice("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890!@#$%^&*()_-+=[]{}|;:,.<>?")
    return ''.join(noisy_text)

class EDPCollatorRobust:
    def __init__(self, text_noise_level=0.0, image_noise=False):
        self.text_noise_level = text_noise_level
        self.image_noise = image_noise
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.feature_extractor = BeitImageProcessor.from_pretrained('microsoft/dit-base')

    def __call__(self, data):
        text_list, picture_list, label_list = zip(*data)
        
        if self.text_noise_level > 0:
            text_list = [add_text_noise(t, self.text_noise_level) for t in text_list]
        if self.image_noise:
            picture_list = [add_gaussian_noise(p) for p in picture_list]

        text_tensor = self.tokenizer(list(text_list), return_tensors='pt', max_length=256, truncation=True, padding='max_length')
        pixel_values = self.feature_extractor(list(picture_list), return_tensors='pt')
        labels = torch.LongTensor(label_list)
        
        inputs = dict()
        inputs.update(text_tensor)
        inputs.update(pixel_values)
        inputs['labels'] = labels
        return inputs

def compute_advanced_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)
    
    return {"accuracy": acc, "f1_score": f1, "precision": precision, "recall": recall}

# =====================================================================
# 🚀 2. TEST LABORATUVARI (RAKİP MODEL: FOLD 4)
# =====================================================================
if __name__ == "__main__":
    print("🚨 RAKİP MODEL (Orijinal MMTD - Fold 4) Detaylı Testi Başlıyor...\n")

    DATA_CSV_YOLU = '/kaggle/working/MMTD/DATA/email_data/EDP.csv' 
    RESIMLERIN_YOLU = '/kaggle/working/MMTD/DATA/email_data/pics'

    df = pd.read_csv(DATA_CSV_YOLU)
    df = df.rename(columns={"texts": "text", "pics": "image", "labels": "label"})
    _, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
    test_dataset = EDPDataset(RESIMLERIN_YOLU, test_df)

    # Orijinal MMTD Modelini Kuruyoruz
    model = MMTD(bert_pretrain_weight='bert-base-multilingual-cased', 
                 beit_pretrain_weight='microsoft/dit-base')
    
    # 🔥 BULDUĞUMUZ DOĞRU YOL BURADA 🔥
    rakip_model_path = '/kaggle/input/datasets/arafkubraa/mmtdcheckpointsfold4/pytorch_model.bin'
    
    print(f"🧠 Ağırlıklar yükleniyor: {rakip_model_path}")
    
    # PyTorch standart yükleme
    model.load_state_dict(torch.load(rakip_model_path))
    model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    model.eval()

    args = TrainingArguments(
        output_dir='./test_results_rakip',
        per_device_eval_batch_size=8,
        report_to="none"
    )

    scenarios = [
        {"name": "Temiz Veri", "text_noise": 0.0, "image_noise": False},
        {"name": "Görüntü Bozuk", "text_noise": 0.0, "image_noise": True},
        {"name": "Metin Bozuk", "text_noise": 0.1, "image_noise": False},
        {"name": "Tam Kaos", "text_noise": 0.1, "image_noise": True},
    ]

    names, accuracies, f1_scores = [], [], []

    print("\n📊 --- ORİJİNAL MMTD KARŞILAŞTIRMALI SONUÇLAR ---")
    for s in scenarios:
        collator = EDPCollatorRobust(text_noise_level=s["text_noise"], image_noise=s["image_noise"])
        trainer = Trainer(
            model=model,
            args=args,
            eval_dataset=test_dataset,
            data_collator=collator,
            compute_metrics=compute_advanced_metrics 
        )
        
        result = trainer.evaluate() 
        acc = result['eval_accuracy'] * 100
        f1 = result['eval_f1_score'] * 100
        
        names.append(s['name'])
        accuracies.append(acc)
        f1_scores.append(f1)
        
        print(f"❌ [{s['name']}] -> Acc: %{acc:.2f} | F1: %{f1:.2f} | Prec: %{result['eval_precision']*100:.2f} | Rec: %{result['eval_recall']*100:.2f}")

    # =====================================================================
    # 🎨 3. RAKİP MODEL İÇİN GRAFİK ÇİZİMİ (Kırmızı Tonlar)
    # =====================================================================
    print("\n🎨 Makale için rakip grafik çiziliyor...")
    x = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#d62728') 
    rects2 = ax.bar(x + width/2, f1_scores, width, label='F1-Score', color='#ff7f0e') 

    ax.set_ylabel('Yüzde (%)', fontsize=12, fontweight='bold')
    ax.set_title('Orijinal MMTD Modeli Gürbüzlük (Robustness) Performansı', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=11)
    ax.set_ylim([0, 110])
    ax.legend(loc='lower left')

    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.1f}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), 
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=10)

    autolabel(rects1)
    autolabel(rects2)
    plt.tight_layout()
    
    plt.savefig('/kaggle/working/mmtd_original_robustness.png', dpi=300)
    plt.show()
    print("✅ Grafik '/kaggle/working/mmtd_original_robustness.png' olarak kaydedildi! İndirebilirsin.")

In [ ]:
import sys
import os
import torch
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

from transformers import (
    Trainer,
    TrainingArguments,
    BertTokenizerFast,
    BeitImageProcessor
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# =====================================================================
# 🛠 1. MİMARİ VE VERİ HAZIRLIĞI
# =====================================================================

sys.path.append('/kaggle/working/MMTD')

# ORİJİNAL MMTD
from models import MMTD
from Email_dataset import EDPDataset


# =====================================================================
# 📌 GÜRÜLTÜ FONKSİYONLARI (DOKUNULMADI)
# =====================================================================

def add_gaussian_noise(image, mean=0, std=25):
    img_array = np.array(image)

    noise = np.random.normal(
        mean,
        std,
        img_array.shape
    ).astype(np.uint8)

    noisy_image_array = np.clip(
        img_array + noise,
        0,
        255
    ).astype(np.uint8)

    return Image.fromarray(noisy_image_array)


def add_text_noise(text, noise_level=0.1):

    if noise_level <= 0 or noise_level > 1:
        return text

    noisy_text = list(text)

    num_noise_chars = int(len(text) * noise_level)

    for _ in range(num_noise_chars):

        index = random.randint(0, len(text) - 1)

        noisy_text[index] = random.choice(
            "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890!@#$%^&*()_-+=[]{}|;:,.<>?"
        )

    return ''.join(noisy_text)


# =====================================================================
# 📌 ROBUST COLLATOR
# =====================================================================

class EDPCollatorRobust:

    def __init__(
        self,
        text_noise_level=0.0,
        image_noise=False
    ):

        self.text_noise_level = text_noise_level
        self.image_noise = image_noise

        self.tokenizer = BertTokenizerFast.from_pretrained(
            'bert-base-multilingual-cased'
        )

        self.feature_extractor = BeitImageProcessor.from_pretrained(
            'microsoft/dit-base'
        )

    def __call__(self, data):

        text_list, picture_list, label_list = zip(*data)

        # -------------------------
        # TEXT NOISE
        # -------------------------
        if self.text_noise_level > 0:
            text_list = [
                add_text_noise(t, self.text_noise_level)
                for t in text_list
            ]

        # -------------------------
        # IMAGE NOISE
        # -------------------------
        if self.image_noise:
            picture_list = [
                add_gaussian_noise(p)
                for p in picture_list
            ]

        # -------------------------
        # TOKENIZATION
        # -------------------------
        text_tensor = self.tokenizer(
            list(text_list),
            return_tensors='pt',
            max_length=256,
            truncation=True,
            padding='max_length'
        )

        # -------------------------
        # IMAGE PROCESSING
        # -------------------------
        pixel_values = self.feature_extractor(
            images=list(picture_list),
            return_tensors='pt'
        )

        labels = torch.LongTensor(label_list)

        inputs = dict()

        inputs.update(text_tensor)
        inputs.update(pixel_values)

        inputs['labels'] = labels

        return inputs


# =====================================================================
# 📌 METRICS
# =====================================================================

def compute_advanced_metrics(eval_pred):

    predictions, labels = eval_pred

    preds = np.argmax(predictions, axis=1)

    acc = accuracy_score(labels, preds)

    f1 = f1_score(
        labels,
        preds,
        average='macro'
    )

    precision = precision_score(
        labels,
        preds,
        average='macro',
        zero_division=0
    )

    recall = recall_score(
        labels,
        preds,
        average='macro',
        zero_division=0
    )

    return {
        "accuracy": acc,
        "f1_score": f1,
        "precision": precision,
        "recall": recall
    }


# =====================================================================
# 🚀 TEST LABORATUVARI
# =====================================================================

if __name__ == "__main__":

    print("\n🚨 ORİJİNAL MMTD ROBUSTNESS TESTİ BAŞLADI\n")

    # ============================================================
    # DATASET
    # ============================================================

    DATA_CSV_YOLU = '/kaggle/working/MMTD/DATA/email_data/EDP.csv'

    RESIMLERIN_YOLU = '/kaggle/working/MMTD/DATA/email_data/pics'

    df = pd.read_csv(DATA_CSV_YOLU)

    df = df.rename(
        columns={
            "texts": "text",
            "pics": "image",
            "labels": "label"
        }
    )

    _, test_df = train_test_split(
        df,
        test_size=0.2,
        stratify=df["label"],
        random_state=42
    )

    test_dataset = EDPDataset(
        RESIMLERIN_YOLU,
        test_df
    )

    # ============================================================
    # MODEL
    # ============================================================

    print("🧠 Orijinal MMTD modeli oluşturuluyor...")

    model = MMTD(
        bert_pretrain_weight='bert-base-multilingual-cased',
        beit_pretrain_weight='microsoft/dit-base'
    )

    # ============================================================
    # CHECKPOINT
    # ============================================================

    rakip_model_path = (
        '/kaggle/input/datasets/arafkubraa/'
        'mmtdcheckpointsfold4/pytorch_model.bin'
    )

    print(f"📦 Checkpoint yükleniyor:\n{rakip_model_path}")

    state_dict = torch.load(
        rakip_model_path,
        map_location="cpu"
    )

    # 🔥 ANA ÇÖZÜM
    missing_keys, unexpected_keys = model.load_state_dict(
        state_dict,
        strict=False
    )

    print("\n✅ Checkpoint yüklendi")

    print("\n📌 Missing Keys:")
    print(missing_keys)

    print("\n📌 Unexpected Keys:")
    print(unexpected_keys)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model.to(device)

    model.eval()

    # ============================================================
    # TRAINER ARGUMENTS
    # ============================================================

    args = TrainingArguments(
        output_dir='./test_results_rakip',
        per_device_eval_batch_size=8,
        report_to="none"
    )

    # ============================================================
    # SCENARIOS
    # ============================================================

    scenarios = [

        {
            "name": "Temiz Veri",
            "text_noise": 0.0,
            "image_noise": False
        },

        {
            "name": "Görüntü Bozuk",
            "text_noise": 0.0,
            "image_noise": True
        },

        {
            "name": "Metin Bozuk",
            "text_noise": 0.1,
            "image_noise": False
        },

        {
            "name": "Tam Kaos",
            "text_noise": 0.1,
            "image_noise": True
        }
    ]

    names = []
    accuracies = []
    f1_scores = []

    print("\n📊 --- ROBUSTNESS SONUÇLARI ---\n")

    for s in scenarios:

        collator = EDPCollatorRobust(
            text_noise_level=s["text_noise"],
            image_noise=s["image_noise"]
        )

        trainer = Trainer(
            model=model,
            args=args,
            eval_dataset=test_dataset,
            data_collator=collator,
            compute_metrics=compute_advanced_metrics
        )

        result = trainer.evaluate()

        acc = result['eval_accuracy'] * 100

        f1 = result['eval_f1_score'] * 100

        names.append(s['name'])
        accuracies.append(acc)
        f1_scores.append(f1)

        print(
            f"❌ [{s['name']}] "
            f"-> Acc: %{acc:.2f} "
            f"| F1: %{f1:.2f} "
            f"| Prec: %{result['eval_precision']*100:.2f} "
            f"| Rec: %{result['eval_recall']*100:.2f}"
        )

    # ============================================================
    # 🎨 GRAFİK
    # ============================================================

    print("\n🎨 Grafik çiziliyor...")

    x = np.arange(len(names))

    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))

    rects1 = ax.bar(
        x - width/2,
        accuracies,
        width,
        label='Accuracy'
    )

    rects2 = ax.bar(
        x + width/2,
        f1_scores,
        width,
        label='F1-Score'
    )

    ax.set_ylabel(
        'Yüzde (%)',
        fontsize=12,
        fontweight='bold'
    )

    ax.set_title(
        'Orijinal MMTD Robustness Performansı',
        fontsize=14,
        fontweight='bold'
    )

    ax.set_xticks(x)

    ax.set_xticklabels(
        names,
        fontsize=11
    )

    ax.set_ylim([0, 110])

    ax.legend(loc='lower left')

    def autolabel(rects):

        for rect in rects:

            height = rect.get_height()

            ax.annotate(
                f'{height:.1f}%',

                xy=(
                    rect.get_x() + rect.get_width() / 2,
                    height
                ),

                xytext=(0, 3),

                textcoords="offset points",

                ha='center',
                va='bottom',
                fontsize=10
            )

    autolabel(rects1)
    autolabel(rects2)

    plt.tight_layout()

    save_path = '/kaggle/working/mmtd_original_robustness.png'

    plt.savefig(
        save_path,
        dpi=300
    )

    plt.show()

    print(f"\n✅ Grafik kaydedildi:\n{save_path}")

In [ ]:
# =====================================================================
# 🔥 BOZUK / EKSİK GÖRSELLERİ OTOMATİK HANDLE ET
# BUNU robustness testinden ÖNCE çalıştır
# =====================================================================

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import os
import sys

sys.path.append('/kaggle/working/MMTD')

import Email_dataset


# Orijinal __getitem__ metodunu patchliyoruz
def safe_getitem(self, item):

    text = self.data.iloc[item, 0]

    pic_path = os.path.join(
        self.data_path,
        self.data.iloc[item, 1]
    )

    label = self.data.iloc[item, 2]

    try:
        pic = Image.open(pic_path).convert("RGB")

    except Exception as e:

        print(f"⚠️ Problemli dosya kullanılamadı:")
        print(pic_path)
        print(f"Hata: {e}")

        # Siyah placeholder image
        pic = Image.new("RGB", (224, 224), (0, 0, 0))

    return text, pic, label


# Dataset class'ı runtime'da değiştiriyoruz
Email_dataset.EDPDataset.__getitem__ = safe_getitem

print("✅ Güvenli dataset patch aktif!")

In [ ]:
import sys
import os
import torch
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

from transformers import (
    Trainer,
    TrainingArguments,
    BertTokenizerFast,
    BeitImageProcessor
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# =====================================================================
# 🛠 1. MİMARİ VE VERİ HAZIRLIĞI
# =====================================================================

sys.path.append('/kaggle/working/MMTD')

# ORİJİNAL MMTD
from models import MMTD
from Email_dataset import EDPDataset


# =====================================================================
# 📌 GÜRÜLTÜ FONKSİYONLARI (DOKUNULMADI)
# =====================================================================

def add_gaussian_noise(image, mean=0, std=25):
    img_array = np.array(image)

    noise = np.random.normal(
        mean,
        std,
        img_array.shape
    ).astype(np.uint8)

    noisy_image_array = np.clip(
        img_array + noise,
        0,
        255
    ).astype(np.uint8)

    return Image.fromarray(noisy_image_array)


def add_text_noise(text, noise_level=0.1):

    if noise_level <= 0 or noise_level > 1:
        return text

    noisy_text = list(text)

    num_noise_chars = int(len(text) * noise_level)

    for _ in range(num_noise_chars):

        index = random.randint(0, len(text) - 1)

        noisy_text[index] = random.choice(
            "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890!@#$%^&*()_-+=[]{}|;:,.<>?"
        )

    return ''.join(noisy_text)


# =====================================================================
# 📌 ROBUST COLLATOR
# =====================================================================

class EDPCollatorRobust:

    def __init__(
        self,
        text_noise_level=0.0,
        image_noise=False
    ):

        self.text_noise_level = text_noise_level
        self.image_noise = image_noise

        self.tokenizer = BertTokenizerFast.from_pretrained(
            'bert-base-multilingual-cased'
        )

        self.feature_extractor = BeitImageProcessor.from_pretrained(
            'microsoft/dit-base'
        )

    def __call__(self, data):

        text_list, picture_list, label_list = zip(*data)

        # -------------------------
        # TEXT NOISE
        # -------------------------
        if self.text_noise_level > 0:
            text_list = [
                add_text_noise(t, self.text_noise_level)
                for t in text_list
            ]

        # -------------------------
        # IMAGE NOISE
        # -------------------------
        if self.image_noise:
            picture_list = [
                add_gaussian_noise(p)
                for p in picture_list
            ]

        # -------------------------
        # TOKENIZATION
        # -------------------------
        text_tensor = self.tokenizer(
            list(text_list),
            return_tensors='pt',
            max_length=256,
            truncation=True,
            padding='max_length'
        )

        # -------------------------
        # IMAGE PROCESSING
        # -------------------------
        pixel_values = self.feature_extractor(
            images=list(picture_list),
            return_tensors='pt'
        )

        labels = torch.LongTensor(label_list)

        inputs = dict()

        inputs.update(text_tensor)
        inputs.update(pixel_values)

        inputs['labels'] = labels

        return inputs


# =====================================================================
# 📌 METRICS
# =====================================================================

def compute_advanced_metrics(eval_pred):

    predictions, labels = eval_pred

    preds = np.argmax(predictions, axis=1)

    acc = accuracy_score(labels, preds)

    f1 = f1_score(
        labels,
        preds,
        average='macro'
    )

    precision = precision_score(
        labels,
        preds,
        average='macro',
        zero_division=0
    )

    recall = recall_score(
        labels,
        preds,
        average='macro',
        zero_division=0
    )

    return {
        "accuracy": acc,
        "f1_score": f1,
        "precision": precision,
        "recall": recall
    }


# =====================================================================
# 🚀 TEST LABORATUVARI
# =====================================================================

if __name__ == "__main__":

    print("\n🚨 ORİJİNAL MMTD ROBUSTNESS TESTİ BAŞLADI\n")

    # ============================================================
    # DATASET
    # ============================================================

    DATA_CSV_YOLU = '/kaggle/working/MMTD/DATA/email_data/EDP.csv'

    RESIMLERIN_YOLU = '/kaggle/working/MMTD/DATA/email_data/pics'

    df = pd.read_csv(DATA_CSV_YOLU)

    df = df.rename(
        columns={
            "texts": "text",
            "pics": "image",
            "labels": "label"
        }
    )

    _, test_df = train_test_split(
        df,
        test_size=0.2,
        stratify=df["label"],
        random_state=42
    )

    test_dataset = EDPDataset(
        RESIMLERIN_YOLU,
        test_df
    )

    # ============================================================
    # MODEL
    # ============================================================

    print("🧠 Orijinal MMTD modeli oluşturuluyor...")

    model = MMTD(
        bert_pretrain_weight='bert-base-multilingual-cased',
        beit_pretrain_weight='microsoft/dit-base'
    )

    # ============================================================
    # CHECKPOINT
    # ============================================================

    rakip_model_path = (
        '/kaggle/input/datasets/arafkubraa/'
        'mmtdcheckpointsfold4/pytorch_model.bin'
    )

    print(f"📦 Checkpoint yükleniyor:\n{rakip_model_path}")

    state_dict = torch.load(
        rakip_model_path,
        map_location="cpu"
    )

    # 🔥 ANA ÇÖZÜM
    missing_keys, unexpected_keys = model.load_state_dict(
        state_dict,
        strict=False
    )

    print("\n✅ Checkpoint yüklendi")

    print("\n📌 Missing Keys:")
    print(missing_keys)

    print("\n📌 Unexpected Keys:")
    print(unexpected_keys)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model.to(device)

    model.eval()

    # ============================================================
    # TRAINER ARGUMENTS
    # ============================================================

    args = TrainingArguments(
        output_dir='./test_results_rakip',
        per_device_eval_batch_size=8,
        report_to="none"
    )

    # ============================================================
    # SCENARIOS
    # ============================================================

    scenarios = [

        {
            "name": "Temiz Veri",
            "text_noise": 0.0,
            "image_noise": False
        },

        {
            "name": "Görüntü Bozuk",
            "text_noise": 0.0,
            "image_noise": True
        },

        {
            "name": "Metin Bozuk",
            "text_noise": 0.1,
            "image_noise": False
        },

        {
            "name": "Tam Kaos",
            "text_noise": 0.1,
            "image_noise": True
        }
    ]

    names = []
    accuracies = []
    f1_scores = []

    print("\n📊 --- ROBUSTNESS SONUÇLARI ---\n")

    for s in scenarios:

        collator = EDPCollatorRobust(
            text_noise_level=s["text_noise"],
            image_noise=s["image_noise"]
        )

        trainer = Trainer(
            model=model,
            args=args,
            eval_dataset=test_dataset,
            data_collator=collator,
            compute_metrics=compute_advanced_metrics
        )

        result = trainer.evaluate()

        acc = result['eval_accuracy'] * 100

        f1 = result['eval_f1_score'] * 100

        names.append(s['name'])
        accuracies.append(acc)
        f1_scores.append(f1)

        print(
            f"❌ [{s['name']}] "
            f"-> Acc: %{acc:.2f} "
            f"| F1: %{f1:.2f} "
            f"| Prec: %{result['eval_precision']*100:.2f} "
            f"| Rec: %{result['eval_recall']*100:.2f}"
        )

    # ============================================================
    # 🎨 GRAFİK
    # ============================================================

    print("\n🎨 Grafik çiziliyor...")

    x = np.arange(len(names))

    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))

    rects1 = ax.bar(
        x - width/2,
        accuracies,
        width,
        label='Accuracy'
    )

    rects2 = ax.bar(
        x + width/2,
        f1_scores,
        width,
        label='F1-Score'
    )

    ax.set_ylabel(
        'Yüzde (%)',
        fontsize=12,
        fontweight='bold'
    )

    ax.set_title(
        'Orijinal MMTD Robustness Performansı',
        fontsize=14,
        fontweight='bold'
    )

    ax.set_xticks(x)

    ax.set_xticklabels(
        names,
        fontsize=11
    )

    ax.set_ylim([0, 110])

    ax.legend(loc='lower left')

    def autolabel(rects):

        for rect in rects:

            height = rect.get_height()

            ax.annotate(
                f'{height:.1f}%',

                xy=(
                    rect.get_x() + rect.get_width() / 2,
                    height
                ),

                xytext=(0, 3),

                textcoords="offset points",

                ha='center',
                va='bottom',
                fontsize=10
            )

    autolabel(rects1)
    autolabel(rects2)

    plt.tight_layout()

    save_path = '/kaggle/working/mmtd_original_robustness.png'

    plt.savefig(
        save_path,
        dpi=300
    )

    plt.show()

    print(f"\n✅ Grafik kaydedildi:\n{save_path}")

In [ ]:
# =====================================================================
# 🔥 TOKENIZER TEXT HATASI ÇÖZÜMÜ
# BUNU robustness testinden ÖNCE çalıştır
# =====================================================================

import sys

sys.path.append('/kaggle/working/MMTD')

from transformers import BertTokenizerFast, BeitImageProcessor
import torch

# Mevcut class'ı import et
from __main__ import EDPCollatorRobust


# Güvenli __call__ override
def safe_call(self, data):

    text_list, picture_list, label_list = zip(*data)

    # -------------------------------------------------
    # TEXT NOISE
    # -------------------------------------------------
    if self.text_noise_level > 0:
        text_list = [
            add_text_noise(str(t), self.text_noise_level)
            for t in text_list
        ]

    # -------------------------------------------------
    # IMAGE NOISE
    # -------------------------------------------------
    if self.image_noise:
        picture_list = [
            add_gaussian_noise(p)
            for p in picture_list
        ]

    # -------------------------------------------------
    # TEXT CLEAN
    # -------------------------------------------------
    clean_texts = [
        str(t) if t is not None else ""
        for t in text_list
    ]

    # -------------------------------------------------
    # TOKENIZATION
    # -------------------------------------------------
    text_tensor = self.tokenizer(
        clean_texts,
        return_tensors='pt',
        max_length=256,
        truncation=True,
        padding='max_length'
    )

    # -------------------------------------------------
    # IMAGE PROCESSING
    # -------------------------------------------------
    pixel_values = self.feature_extractor(
        list(picture_list),
        return_tensors='pt'
    )

    labels = torch.LongTensor(label_list)

    inputs = dict()

    inputs.update(text_tensor)
    inputs.update(pixel_values)

    inputs['labels'] = labels

    return inputs


# Runtime patch
EDPCollatorRobust.__call__ = safe_call

print("✅ Tokenizer güvenlik patch aktif!")

In [ ]:
import sys
import os
import torch
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import Trainer, TrainingArguments, BertTokenizerFast, BeitImageProcessor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import Dataset

# =====================================================================
# 🛠 1. MİMARİ YÜKLEMESİ
# =====================================================================
sys.path.append('/kaggle/working/MMTD')
from models import MMTD  # Orijinal mimari

# =====================================================================
# 🛡️ 2. KURŞUN GEÇİRMEZ (BULLETPROOF) VERİ SETİ SINIFI
# Hata 2 (Çince Resimler) ve Hata 3'ü (NaN Metinler) çözer.
# =====================================================================
class EDPDatasetBulletproof(Dataset):
    def __init__(self, img_dir, dataframe):
        self.img_dir = img_dir
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # 1. Metin Koruması (NaN değerleri temizle)
        raw_text = self.dataframe.loc[idx, 'text']
        text = str(raw_text) if pd.notna(raw_text) else ""
        if text.strip().lower() == 'nan': 
            text = ""
            
        # 2. Resim Koruması (Bulunamazsa siyah resim ver)
        img_name = str(self.dataframe.loc[idx, 'image'])
        img_path = os.path.join(self.img_dir, img_name)
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        label = int(self.dataframe.loc[idx, 'label'])
        return text, image, label

# =====================================================================
# 🌪️ 3. GÜRÜLTÜ VE COLLATOR
# =====================================================================
def add_gaussian_noise(image, mean=0, std=25):
    img_array = np.array(image)
    noise = np.random.normal(mean, std, img_array.shape).astype(np.uint8)
    noisy_image_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_image_array)

def add_text_noise(text, noise_level=0.1):
    if noise_level <= 0 or noise_level > 1 or len(text) == 0: return text
    noisy_text = list(text)
    num_noise_chars = int(len(text) * noise_level)
    for _ in range(num_noise_chars):
        index = random.randint(0, len(text) - 1)
        noisy_text[index] = random.choice("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890!@#$%^&*()_-+=[]{}|;:,.<>?")
    return ''.join(noisy_text)

class EDPCollatorRobust:
    def __init__(self, text_noise_level=0.0, image_noise=False):
        self.text_noise_level = text_noise_level
        self.image_noise = image_noise
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.feature_extractor = BeitImageProcessor.from_pretrained('microsoft/dit-base')

    def __call__(self, data):
        text_list, picture_list, label_list = zip(*data)
        
        if self.text_noise_level > 0:
            text_list = [add_text_noise(t, self.text_noise_level) for t in text_list]
        if self.image_noise:
            picture_list = [add_gaussian_noise(p) for p in picture_list]

        text_tensor = self.tokenizer(list(text_list), return_tensors='pt', max_length=256, truncation=True, padding='max_length')
        pixel_values = self.feature_extractor(list(picture_list), return_tensors='pt')
        labels = torch.LongTensor(label_list)
        
        inputs = dict()
        inputs.update(text_tensor)
        inputs.update(pixel_values)
        inputs['labels'] = labels
        return inputs

def compute_advanced_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)
    return {"accuracy": acc, "f1_score": f1, "precision": precision, "recall": recall}

# =====================================================================
# 🚀 4. ANA TEST DÖNGÜSÜ
# =====================================================================
if __name__ == "__main__":
    print("🚨 RAKİP MODEL (Orijinal MMTD) Kurşun Geçirmez Testi Başlıyor...\n")

    DATA_CSV_YOLU = '/kaggle/working/MMTD/DATA/email_data/EDP.csv' 
    RESIMLERIN_YOLU = '/kaggle/working/MMTD/DATA/email_data/pics'

    df = pd.read_csv(DATA_CSV_YOLU)
    df = df.rename(columns={"texts": "text", "pics": "image", "labels": "label"})
    _, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
    
    # ⚠️ Kendi yazdığımız sağlam Dataset'i kullanıyoruz
    test_dataset = EDPDatasetBulletproof(RESIMLERIN_YOLU, test_df)

    model = MMTD(bert_pretrain_weight='bert-base-multilingual-cased', 
                 beit_pretrain_weight='microsoft/dit-base')
    
    rakip_model_path = '/kaggle/input/datasets/arafkubraa/mmtdcheckpointsfold4/pytorch_model.bin'
    print(f"🧠 Ağırlıklar yükleniyor (strict=False ile): {rakip_model_path}")
    
    # ⚠️ Hata 1'i (Unexpected keys) çözen kritik ekleme: strict=False
    model.load_state_dict(torch.load(rakip_model_path), strict=False)
    
    model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    model.eval()

    args = TrainingArguments(output_dir='./test_results_rakip', per_device_eval_batch_size=8, report_to="none")

    scenarios = [
        {"name": "Temiz Veri", "text_noise": 0.0, "image_noise": False},
        {"name": "Görüntü Bozuk", "text_noise": 0.0, "image_noise": True},
        {"name": "Metin Bozuk", "text_noise": 0.1, "image_noise": False},
        {"name": "Tam Kaos", "text_noise": 0.1, "image_noise": True},
    ]

    names, accuracies, f1_scores = [], [], []

    for s in scenarios:
        collator = EDPCollatorRobust(text_noise_level=s["text_noise"], image_noise=s["image_noise"])
        trainer = Trainer(model=model, args=args, eval_dataset=test_dataset, data_collator=collator, compute_metrics=compute_advanced_metrics)
        
        result = trainer.evaluate() 
        acc = result['eval_accuracy'] * 100
        f1 = result['eval_f1_score'] * 100
        
        names.append(s['name'])
        accuracies.append(acc)
        f1_scores.append(f1)
        
        print(f"❌ [{s['name']}] -> Acc: %{acc:.2f} | F1: %{f1:.2f} | Prec: %{result['eval_precision']*100:.2f} | Rec: %{result['eval_recall']*100:.2f}")

    # Grafik Çizimi
    x = np.arange(len(names))
    width = 0.35
    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#d62728') 
    rects2 = ax.bar(x + width/2, f1_scores, width, label='F1-Score', color='#ff7f0e') 

    ax.set_ylabel('Yüzde (%)', fontsize=12, fontweight='bold')
    ax.set_title('Orijinal MMTD Modeli Gürbüzlük (Robustness) Performansı', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=11)
    ax.set_ylim([0, 110])
    ax.legend(loc='lower left')

    for rects in [rects1, rects2]:
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.1f}%', xy=(rect.get_x() + rect.get_width() / 2, height), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig('/kaggle/working/mmtd_original_robustness.png', dpi=300)
    plt.show()
    print("✅ Bütün engeller aşıldı! Grafik kaydedildi.")

In [ ]:
import sys
import os
import torch
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import Trainer, TrainingArguments, BertTokenizerFast, BeitImageProcessor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import Dataset

# =====================================================================
# 🛠 1. MİMARİ YÜKLEMESİ (ORİJİNAL MMTD)
# =====================================================================
sys.path.append('/kaggle/working/MMTD')
from models import MMTD 

# =====================================================================
# 🛡️ 2. KURŞUN GEÇİRMEZ (BULLETPROOF) VERİ SETİ SINIFI
# =====================================================================
class EDPDatasetBulletproof(Dataset):
    def __init__(self, img_dir, dataframe):
        self.img_dir = img_dir
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # 1. Metin Koruması (NaN ve Boş Değerler)
        raw_text = self.dataframe.loc[idx, 'text']
        text = str(raw_text) if pd.notna(raw_text) else ""
        if text.strip().lower() == 'nan': 
            text = ""
            
        # 2. Resim Koruması (Çince, bozuk veya eksik dosyalar)
        img_name = str(self.dataframe.loc[idx, 'image'])
        img_path = os.path.join(self.img_dir, img_name)
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        label = int(self.dataframe.loc[idx, 'label'])
        return text, image, label

# =====================================================================
# 🌪️ 3. GÜRÜLTÜ ÜRETİCİ VE BİRLEŞTİRİCİ (COLLATOR)
# =====================================================================
def add_gaussian_noise(image, mean=0, std=25):
    img_array = np.array(image)
    noise = np.random.normal(mean, std, img_array.shape).astype(np.uint8)
    noisy_image_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_image_array)

def add_text_noise(text, noise_level=0.1):
    if noise_level <= 0 or noise_level > 1 or len(text) == 0: return text
    noisy_text = list(text)
    num_noise_chars = int(len(text) * noise_level)
    for _ in range(num_noise_chars):
        index = random.randint(0, len(text) - 1)
        noisy_text[index] = random.choice("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890!@#$%^&*()_-+=[]{}|;:,.<>?")
    return ''.join(noisy_text)

class EDPCollatorRobust:
    def __init__(self, text_noise_level=0.0, image_noise=False):
        self.text_noise_level = text_noise_level
        self.image_noise = image_noise
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.feature_extractor = BeitImageProcessor.from_pretrained('microsoft/dit-base')

    def __call__(self, data):
        text_list, picture_list, label_list = zip(*data)
        
        if self.text_noise_level > 0:
            text_list = [add_text_noise(t, self.text_noise_level) for t in text_list]
        if self.image_noise:
            picture_list = [add_gaussian_noise(p) for p in picture_list]

        text_tensor = self.tokenizer(list(text_list), return_tensors='pt', max_length=256, truncation=True, padding='max_length')
        pixel_values = self.feature_extractor(list(picture_list), return_tensors='pt')
        labels = torch.LongTensor(label_list)
        
        inputs = dict()
        inputs.update(text_tensor)
        inputs.update(pixel_values)
        inputs['labels'] = labels
        return inputs

def compute_advanced_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)
    return {"accuracy": acc, "f1_score": f1, "precision": precision, "recall": recall}

# =====================================================================
# 🚀 4. ANA TEST DÖNGÜSÜ
# =====================================================================
if __name__ == "__main__":
    print("🚨 RAKİP MODEL (Orijinal MMTD) Kurşun Geçirmez Testi Başlıyor...\n")

    DATA_CSV_YOLU = '/kaggle/working/MMTD/DATA/email_data/EDP.csv' 
    RESIMLERIN_YOLU = '/kaggle/working/MMTD/DATA/email_data/pics'

    df = pd.read_csv(DATA_CSV_YOLU)
    df = df.rename(columns={"texts": "text", "pics": "image", "labels": "label"})
    _, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
    
    # 🛡️ Kendi yazdığımız sağlam Dataset'i kullanıyoruz
    test_dataset = EDPDatasetBulletproof(RESIMLERIN_YOLU, test_df)

    print("🧠 Orijinal MMTD Mimarisi Çağrılıyor...")
    model = MMTD(bert_pretrain_weight='bert-base-multilingual-cased', 
                 beit_pretrain_weight='microsoft/dit-base')
    
    # 📍 KESİNLEŞMİŞ DOSYA YOLU
    rakip_model_path = '/kaggle/input/datasets/arafkubraa/mmtdcheckpointsfold4/pytorch_model.bin'
    print(f"📥 Ağırlıklar yükleniyor (strict=False ile): {rakip_model_path}")
    
    # ⚠️ Hata 1'i (Unexpected keys) çözen kritik ekleme: strict=False
    model.load_state_dict(torch.load(rakip_model_path), strict=False)
    model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    model.eval()

    # ⚠️ OOM (Bellek Taşkını) İÇİN NİHAİ ZIRH
    args = TrainingArguments(
        output_dir='./test_results_rakip', 
        per_device_eval_batch_size=2,   # Lokmayı iyice küçülttük (4'ten 2'ye)
        fp16=True,                      # Yarı hassasiyet açık
        eval_accumulation_steps=5,      # 🔥 İŞTE OOM'UN KATİLİ BU AYAR! Her 5 adımda bir GPU'yu boşaltır.
        report_to="none"
    )

    scenarios = [
        {"name": "Temiz Veri", "text_noise": 0.0, "image_noise": False},
        {"name": "Görüntü Bozuk", "text_noise": 0.0, "image_noise": True},
        {"name": "Metin Bozuk", "text_noise": 0.1, "image_noise": False},
        {"name": "Tam Kaos", "text_noise": 0.1, "image_noise": True},
    ]

    names, accuracies, f1_scores = [], [], []

    for s in scenarios:
        # 🧹 Her senaryo öncesi ekran kartı belleğini temizle
        torch.cuda.empty_cache()
        
        collator = EDPCollatorRobust(text_noise_level=s["text_noise"], image_noise=s["image_noise"])
        trainer = Trainer(
            model=model, 
            args=args, 
            eval_dataset=test_dataset, 
            data_collator=collator, 
            compute_metrics=compute_advanced_metrics
        )
        
        result = trainer.evaluate() 
        acc = result['eval_accuracy'] * 100
        f1 = result['eval_f1_score'] * 100
        
        names.append(s['name'])
        accuracies.append(acc)
        f1_scores.append(f1)
        
        print(f"❌ [{s['name']}] -> Acc: %{acc:.2f} | F1: %{f1:.2f} | Prec: %{result['eval_precision']*100:.2f} | Rec: %{result['eval_recall']*100:.2f}")

    # =====================================================================
    # 🎨 5. GRAFİK ÇİZİMİ
    # =====================================================================
    print("\n🎨 Makale için grafik çiziliyor...")
    x = np.arange(len(names))
    width = 0.35
    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#d62728') 
    rects2 = ax.bar(x + width/2, f1_scores, width, label='F1-Score', color='#ff7f0e') 

    ax.set_ylabel('Yüzde (%)', fontsize=12, fontweight='bold')
    ax.set_title('Orijinal MMTD Modeli Gürbüzlük (Robustness) Performansı', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=11)
    ax.set_ylim([0, 110])
    ax.legend(loc='lower left')

    for rects in [rects1, rects2]:
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.1f}%', xy=(rect.get_x() + rect.get_width() / 2, height), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig('/kaggle/working/mmtd_original_robustness.png', dpi=300)
    plt.show()
    print("✅ Bütün engeller aşıldı! Grafik kaydedildi.")

In [ ]:
import sys
import os
import torch
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import BertTokenizerFast, BeitImageProcessor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm # İlerleme çubuğu için

# =====================================================================
# 🛠 1. MİMARİ YÜKLEMESİ
# =====================================================================
sys.path.append('/kaggle/working/MMTD')
from models import MMTD 

# =====================================================================
# 🛡️ 2. KURŞUN GEÇİRMEZ VERİ SETİ
# =====================================================================
class EDPDatasetBulletproof(Dataset):
    def __init__(self, img_dir, dataframe):
        self.img_dir = img_dir
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        raw_text = self.dataframe.loc[idx, 'text']
        text = str(raw_text) if pd.notna(raw_text) else ""
        if text.strip().lower() == 'nan': text = ""
            
        img_name = str(self.dataframe.loc[idx, 'image'])
        img_path = os.path.join(self.img_dir, img_name)
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        label = int(self.dataframe.loc[idx, 'label'])
        return text, image, label

# =====================================================================
# 🌪️ 3. GÜRÜLTÜ VE COLLATOR
# =====================================================================
def add_gaussian_noise(image, mean=0, std=25):
    img_array = np.array(image)
    noise = np.random.normal(mean, std, img_array.shape).astype(np.uint8)
    return Image.fromarray(np.clip(img_array + noise, 0, 255).astype(np.uint8))

def add_text_noise(text, noise_level=0.1):
    if noise_level <= 0 or noise_level > 1 or len(text) == 0: return text
    noisy_text = list(text)
    num_noise_chars = int(len(text) * noise_level)
    for _ in range(num_noise_chars):
        noisy_text[random.randint(0, len(text) - 1)] = random.choice("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890!@#$%^&*()")
    return ''.join(noisy_text)

class EDPCollatorRobust:
    def __init__(self, text_noise_level=0.0, image_noise=False):
        self.text_noise_level = text_noise_level
        self.image_noise = image_noise
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.feature_extractor = BeitImageProcessor.from_pretrained('microsoft/dit-base')

    def __call__(self, data):
        text_list, picture_list, label_list = zip(*data)
        if self.text_noise_level > 0:
            text_list = [add_text_noise(t, self.text_noise_level) for t in text_list]
        if self.image_noise:
            picture_list = [add_gaussian_noise(p) for p in picture_list]

        text_tensor = self.tokenizer(list(text_list), return_tensors='pt', max_length=256, truncation=True, padding='max_length')
        pixel_values = self.feature_extractor(list(picture_list), return_tensors='pt')
        
        inputs = dict(text_tensor)
        inputs.update(pixel_values)
        inputs['labels'] = torch.LongTensor(label_list)
        return inputs

# =====================================================================
# 🔥 4. OOM KATİLİ: SAF PYTORCH TEST DÖNGÜSÜ
# =====================================================================
def evaluate_custom(model, dataset, collator, device):
    # Batch Size = 1: VRAM taşkınını matematiksel olarak imkansız kılar
    dataloader = DataLoader(dataset, batch_size=1, collate_fn=collator, shuffle=False)
    
    all_preds, all_labels = [], []
    
    with torch.no_grad(): # Hafıza istiflemesini tamamen kapatır
        for batch in tqdm(dataloader, desc="Test Ediliyor"):
            inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
            labels = batch['labels'].to(device)

            with torch.cuda.amp.autocast(): # Yarı hassasiyet (FP16)
                outputs = model(**inputs)

            # Çıktı türüne göre logitleri al
            logits = outputs[0] if isinstance(outputs, tuple) else (outputs.get('logits', outputs) if isinstance(outputs, dict) else outputs)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            # GPU'yu her adımda zorla ve acımasızca temizle
            del inputs, labels, outputs, logits, preds
            torch.cuda.empty_cache()

    acc = accuracy_score(all_labels, all_preds) * 100
    f1 = f1_score(all_labels, all_preds, average='macro') * 100
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0) * 100
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0) * 100
    return acc, f1, prec, rec

# =====================================================================
# 🚀 5. ANA İŞLETİM
# =====================================================================
if __name__ == "__main__":
    print("🚨 RAKİP MODEL (Orijinal MMTD) Saf PyTorch Testi Başlıyor...\n")

    DATA_CSV_YOLU = '/kaggle/working/MMTD/DATA/email_data/EDP.csv' 
    RESIMLERIN_YOLU = '/kaggle/working/MMTD/DATA/email_data/pics'

    df = pd.read_csv(DATA_CSV_YOLU)
    df = df.rename(columns={"texts": "text", "pics": "image", "labels": "label"})
    _, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
    
    test_dataset = EDPDatasetBulletproof(RESIMLERIN_YOLU, test_df)

    print("🧠 Orijinal MMTD Mimarisi Çağrılıyor...")
    model = MMTD(bert_pretrain_weight='bert-base-multilingual-cased', beit_pretrain_weight='microsoft/dit-base')
    
    rakip_model_path = '/kaggle/input/datasets/arafkubraa/mmtdcheckpointsfold4/pytorch_model.bin'
    print(f"📥 Ağırlıklar yükleniyor...")
    model.load_state_dict(torch.load(rakip_model_path), strict=False)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    scenarios = [
        {"name": "Temiz Veri", "text_noise": 0.0, "image_noise": False},
        {"name": "Görüntü Bozuk", "text_noise": 0.0, "image_noise": True},
        {"name": "Metin Bozuk", "text_noise": 0.1, "image_noise": False},
        {"name": "Tam Kaos", "text_noise": 0.1, "image_noise": True},
    ]

    names, accuracies, f1_scores = [], [], []

    for s in scenarios:
        torch.cuda.empty_cache() # Her senaryo öncesi RAM'i yıka
        print(f"\n▶️ SENARYO BAŞLIYOR: {s['name']}")
        
        collator = EDPCollatorRobust(text_noise_level=s["text_noise"], image_noise=s["image_noise"])
        
        # 🔥 HuggingFace Trainer yerine KENDİ DÖNGÜMÜZÜ çağırıyoruz
        acc, f1, prec, rec = evaluate_custom(model, test_dataset, collator, device)
        
        names.append(s['name'])
        accuracies.append(acc)
        f1_scores.append(f1)
        print(f"✅ [{s['name']}] -> Acc: %{acc:.2f} | F1: %{f1:.2f} | Prec: %{prec:.2f} | Rec: %{rec:.2f}")

    # Grafik Çizimi
    print("\n🎨 Grafik çiziliyor...")
    x = np.arange(len(names))
    width = 0.35
    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#d62728') 
    rects2 = ax.bar(x + width/2, f1_scores, width, label='F1-Score', color='#ff7f0e') 

    ax.set_ylabel('Yüzde (%)', fontsize=12, fontweight='bold')
    ax.set_title('Orijinal MMTD Modeli Gürbüzlük (Robustness) Performansı', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=11)
    ax.set_ylim([0, 110])
    ax.legend(loc='lower left')

    for rects in [rects1, rects2]:
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.1f}%', xy=(rect.get_x() + rect.get_width() / 2, height), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig('/kaggle/working/mmtd_original_robustness.png', dpi=300)
    plt.show()